<a href="https://colab.research.google.com/github/gcc712-lgtm/bigdata/blob/main/week04_5756300_%EA%B9%80%EC%B1%84%EC%9B%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:김채원**
**학번:5756300**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [ ]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cu130 / device: cuda


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [ ]:
# 답안

tensors = {
    # 클래스 인덱스와 같은 정수형 데이터를 표현하기 생성
    "a 정수": torch.tensor([1, 2, 3]),

    # 연속형 입력 데이터나 매개변수와 같은 실수형 데이터를 표현하기 위해 생성
    "b 실수":   torch.tensor([1.0, 2.0, 3.0]),

    # 초기값을 0으로 설정해야 하는 데이터나 모델의 계산 공간을 만들기 위해 생성
    "c 영행렬": torch.zeros(2, 3),

    # 다양한 난수 데이터를 활용해 모델의 초기 가중치나 무작위 입력을 실험하기 위해 생성
    "d 난수":   torch.randn(2, 3, 4, 5),

    # 손실 함수의 결과값이나 스칼라 상수를 표현하기 위해 생성
    "e 스칼라": torch.tensor(6.0),
}

rows = []
for name, t in tensors.items():
    rows.append({"이름": name, "shape": tuple(t.shape), "dim": t.dim(),
                 "dtype": str(t.dtype), "device": str(t.device) })

pd.DataFrame(rows)

,이름,shape,dim,dtype,device
0,a 정수,"(3,)",1,torch.int64,cpu
1,b 실수,"(3,)",1,torch.float32,cpu
2,c 영행렬,"(2, 3)",2,torch.float32,cpu
3,d 난수,"(2, 3, 4, 5)",4,torch.float32,cpu
4,e 스칼라,(),0,torch.float32,cpu


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [ ]:
# 답안

a = torch.tensor([[1, 2, 3, 4],
                  [5, 6, 7, 8],
                  [9, 10, 11, 12]], dtype = torch.float32)

b = torch.tensor([[10, 20],
                 [30, 40],
                 [50, 60],
                 [70, 80]], dtype = torch.float32)

c = a @ b
print("행렬곱 결과 :", c)
print("shape :", c.shape)

print("열별 평균 :", a.mean(dim=0))
print("행별 합계 :", a.sum(dim=1))

행렬곱 결과 : tensor([[ 500.,  600.],
        [1140., 1400.],
        [1780., 2200.]])
shape : torch.Size([3, 2])
열별 평균 : tensor([5., 6., 7., 8.])
행별 합계 : tensor([10., 26., 42.])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [ ]:
# 답안

import time

def bench(device, n=2000, repeat=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeat):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()      # GPU 계산이 끝날 때까지 기다린다
    return (time.time() - start) / repeat

print(f"CPU : {bench('cpu'):.4f} 초")
if torch.cuda.is_available():
    print(f"GPU : {bench('cuda'):.4f} 초")
else:
    print("GPU를 사용할 수 없습니다. CPU에서만 실행")

CPU : 0.1978 초
GPU : 0.0059 초


GPU는 병렬 연산에 최적화되어 있어 직렬 연산을 처리하는 CPU보다 약 33.5배 빠르다.



## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [ ]:
# 답안

# 1번
x = torch.tensor(3.0, requires_grad=True)
y = 2 * (x**3) + 5 * x
y.backward()

print("1번 자동미분 계산 값:", x.grad.item())


# 2번
w = torch.tensor(2.0, requires_grad=True)
z = 3 * (w**4) - 6 * (w**2)
z.backward()

print("2번 자동미분 계산 값:", w.grad.item())

1번 자동미분 계산 값: 59.0
2번 자동미분 계산 값: 72.0


1번 문제와 2번 문제 모두 자동미분 계산 값과 손으로 구한 값의 결과가 일치하다.